<a href="https://colab.research.google.com/github/Nourhasann/CO2-Emission-by-Vehicles/blob/main/Twitter_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.4 MB/s eta 0:00:00


In [2]:
import nltk
!pip install emoji
!pip install nltk
nltk.download("punkt_tab")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download('averaged_perceptron_tagger_eng')
!pip install gensim
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 13.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import re
import numpy as np
import emoji
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.tag import pos_tag
from nltk.stem import PorterStemmer, WordNetLemmatizer
from gensim.models import Word2Vec
from transformers import pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import  DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [4]:
df = pd.read_csv("/content/twitter_training.csv")

# EDA

In [5]:
df.isna().sum()

,0
2401,0
Borderlands,0
Positive,0
"im getting on borderlands and i will murder you all ,",686


In [6]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [7]:
df.drop(["Borderlands","2401"], inplace=True,axis=1)


In [8]:
df.tail(2)

,Positive,"im getting on borderlands and i will murder you all ,"
74679,Positive,Just realized between the windows partition of...
74680,Positive,Just like the windows partition of my Mac is l...


In [9]:
df.rename(columns={"Positive": "sentiment", "im getting on borderlands and i will murder you all ,": "tweet"}, inplace=True)

In [10]:
df.head()

,sentiment,tweet
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


# Data preprocessing

In [11]:
# Text Cleaning : -

def clean_text(text):
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+',' ',text).strip()
    return text

df['tweet'] = df['tweet'].apply(clean_text)
df.tail(2)

,sentiment,tweet
74679,Positive,just realized between the windows partition of...
74680,Positive,just like the windows partition of my mac is l...


# Tokenization and text preprocessing

In [12]:
df["tokenized_tweet"] = df["tweet"].apply(word_tokenize)


In [13]:
stop_words = set(stopwords.words("english"))
df["fillterd_tweets"] = df["tokenized_tweet"].apply(
    lambda words : [word for word in words if word not in stop_words]
)
df.tail(3)

,sentiment,tweet,tokenized_tweet,fillterd_tweets
74678,Positive,just realized the windows partition of my mac ...,"[just, realized, the, windows, partition, of, ...","[realized, windows, partition, mac, years, beh..."
74679,Positive,just realized between the windows partition of...,"[just, realized, between, the, windows, partit...","[realized, windows, partition, mac, like, year..."
74680,Positive,just like the windows partition of my mac is l...,"[just, like, the, windows, partition, of, my, ...","[like, windows, partition, mac, like, years, b..."


**Lemmatization**

In [14]:
"""def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

wl = WordNetLemmatizer()

def lemmatize_with_pos(word_list):
    pos_tagged_words = pos_tag(word_list)
    lemmas = []
    for word, tag in pos_tagged_words:
        wn_tag = get_wordnet_pos(tag)
        lemmas.append(wl.lemmatize(word, wn_tag))
    return lemmas

df['lemmatized_with_pos'] = df['fillterd_tweets'].apply(lemmatize_with_pos)

# Displaying the difference for 'realized' and 'unnoticed'
df[df['tweet'].str.contains('unnoticed|realized', regex=True, case=False)][['tweet', 'fillterd_tweets', 'lemmatized_with_pos']].head(10)"""

"def get_wordnet_pos(tag):\n    if tag.startswith('J'):\n        return wordnet.ADJ\n    elif tag.startswith('V'):\n        return wordnet.VERB\n    elif tag.startswith('N'):\n        return wordnet.NOUN\n    elif tag.startswith('R'):\n        return wordnet.ADV\n    else:\n        return wordnet.NOUN\n\nwl = WordNetLemmatizer()\n\ndef lemmatize_with_pos(word_list):\n    pos_tagged_words = pos_tag(word_list)\n    lemmas = []\n    for word, tag in pos_tagged_words:\n        wn_tag = get_wordnet_pos(tag)\n        lemmas.append(wl.lemmatize(word, wn_tag))\n    return lemmas\n\ndf['lemmatized_with_pos'] = df['fillterd_tweets'].apply(lemmatize_with_pos)\n\n# Displaying the difference for 'realized' and 'unnoticed'\ndf[df['tweet'].str.contains('unnoticed|realized', regex=True, case=False)][['tweet', 'fillterd_tweets', 'lemmatized_with_pos']].head(10)"

**stemming**

In [15]:
"""ps = PorterStemmer()

def stem_words(word_list):
    return [ps.stem(word) for word in word_list]

df['stemmed_tweets'] = df['fillterd_tweets'].apply(stem_words)
df.tail(2)"""

"ps = PorterStemmer()\n\ndef stem_words(word_list):\n    return [ps.stem(word) for word in word_list]\n\ndf['stemmed_tweets'] = df['fillterd_tweets'].apply(stem_words)\ndf.tail(2)"

# word embedding (text representation)

In [16]:
# Initialize and train the Word2Vec model with increased dimensions to reduce loss
corpus =df['fillterd_tweets'].tolist()

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4
)

In [17]:
def get_sentence_vector(word_list, model):
    vectors = [
        model.wv[word] for word in word_list if word in model.wv
    ]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

df['sentence_vector'] = df['fillterd_tweets'].apply(lambda x: get_sentence_vector(x, w2v_model))
df.head()

,sentiment,tweet,tokenized_tweet,fillterd_tweets,sentence_vector
0,Positive,i am coming to the borders and i will kill you...,"[i, am, coming, to, the, borders, and, i, will...","[coming, borders, kill]","[0.048067123, 0.3071308, -0.15533127, -0.29827..."
1,Positive,im getting on borderlands and i will kill you all,"[im, getting, on, borderlands, and, i, will, k...","[im, getting, borderlands, kill]","[-0.08536747, 0.74551296, -0.3665331, -0.70301..."
2,Positive,im coming on borderlands and i will murder you...,"[im, coming, on, borderlands, and, i, will, mu...","[im, coming, borderlands, murder]","[0.05657492, 0.7850465, -0.28415737, -0.503259..."
3,Positive,im getting on borderlands and i will murder yo...,"[im, getting, on, borderlands, and, i, will, m...","[im, getting, borderlands, murder]","[-0.091118604, 0.71212, -0.337805, -0.6228695,..."
4,Positive,im getting into borderlands and i can murder y...,"[im, getting, into, borderlands, and, i, can, ...","[im, getting, borderlands, murder]","[-0.091118604, 0.71212, -0.337805, -0.6228695,..."


In [18]:
la= LabelEncoder()
df["sentiment"] = la.fit_transform(df["sentiment"])

In [19]:
x = df["sentence_vector"]
y = df["sentiment"]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.15, random_state=42)

In [20]:
# Convert Series of arrays to 2D numpy array for compatibility with sklearn
X_train_full = np.stack(xtrain.values)
X_test_final = np.stack(xtest.values)

# Split the training data further into a final training set and a validation set
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_full, ytrain, test_size=0.15, random_state=42, stratify=ytrain
)

print(f"Shape of X_train_final: {X_train_final.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of X_test_final: {X_test_final.shape}")
print(f"Shape of y_train_final: {y_train_final.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of y_test: {ytest.shape}")

Shape of X_train_final: (53956, 200)
Shape of X_val: (9522, 200)
Shape of X_test_final: (11203, 200)
Shape of y_train_final: (53956,)
Shape of y_val: (9522,)
Shape of y_test: (11203,)


## Model Training with Validation Set

In [21]:
from sklearn.ensemble import RandomForestClassifier

models = [
    LogisticRegression(max_iter=1000, random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    GaussianNB()
]

In [22]:
for model in models:
    model_name = model.__class__.__name__

    # Train the model on the final training set
    model.fit(X_train_final, y_train_final)

    # Evaluate on the training set
    train_score = model.score(X_train_final, y_train_final)

    # Evaluate on the validation set (used for hyperparameter tuning during development)
    val_score = model.score(X_val, y_val)

    # Evaluate on the test set (final, unbiased evaluation)
    test_score = model.score(X_test_final, ytest)

    print(f"--- {model_name} ---")
    print(f"{model_name} Training Score: {train_score:.4f}")
    print(f"{model_name} Validation Score: {val_score:.4f}")
    print(f"{model_name} Test Score: {test_score:.4f}\n")

--- LogisticRegression ---
LogisticRegression Training Score: 0.5238
LogisticRegression Validation Score: 0.5189
LogisticRegression Test Score: 0.5224

--- KNeighborsClassifier ---
KNeighborsClassifier Training Score: 0.8263
KNeighborsClassifier Validation Score: 0.7184
KNeighborsClassifier Test Score: 0.7139

--- DecisionTreeClassifier ---
DecisionTreeClassifier Training Score: 0.9675
DecisionTreeClassifier Validation Score: 0.6000
DecisionTreeClassifier Test Score: 0.5883

--- RandomForestClassifier ---
RandomForestClassifier Training Score: 0.9675
RandomForestClassifier Validation Score: 0.7269
RandomForestClassifier Test Score: 0.7262

--- GaussianNB ---
GaussianNB Training Score: 0.4298
GaussianNB Validation Score: 0.4220
GaussianNB Test Score: 0.4342



## Model Training and Evaluation

In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the K-Nearest Neighbors Classifier
knn_model = KNeighborsClassifier()

# Train the model
print("Training K-Nearest Neighbors...")
knn_model.fit(X_train_final, y_train_final)

# Make predictions
y_pred = knn_model.predict(X_test_final)

# Evaluate the model
train_acc = knn_model.score(X_train_final, y_train_final)
test_acc = knn_model.score(X_test_final, ytest)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Detailed report
print("\nClassification Report:")
print(classification_report(ytest, y_pred, target_names=la.classes_))

Training K-Nearest Neighbors...
Training Accuracy: 0.8263
Test Accuracy: 0.7139

Classification Report:
              precision    recall  f1-score   support

  Irrelevant       0.64      0.69      0.66      1996
    Negative       0.75      0.79      0.76      3379
     Neutral       0.71      0.65      0.68      2649
    Positive       0.73      0.70      0.72      3179

    accuracy                           0.71     11203
   macro avg       0.71      0.71      0.71     11203
weighted avg       0.71      0.71      0.71     11203



In [24]:
df["prediction"] = knn_model.predict(np.stack(df["sentence_vector"].values))
df.tail(20)

,sentiment,tweet,tokenized_tweet,fillterd_tweets,sentence_vector,prediction
74661,2,nvidia therefore doesn t want to give up its c...,"[nvidia, therefore, doesn, t, want, to, give, ...","[nvidia, therefore, want, give, crypto, craze,...","[-0.13441725, 0.08962629, -0.05373971, -0.0716...",2
74662,2,is doesnt should i give up its password crypto...,"[is, doesnt, should, i, give, up, its, passwor...","[doesnt, give, password, crypto, wallet, docs,...","[-0.012885453, -0.055725433, -0.021283755, -0....",2
74663,1,nvidia really delayed the weeks,"[nvidia, really, delayed, the, weeks]","[nvidia, really, delayed, weeks]","[-0.10580146, 0.54291755, -0.06625827, -0.3755...",1
74664,1,nvidia really delayed the by weeks,"[nvidia, really, delayed, the, by, weeks]","[nvidia, really, delayed, weeks]","[-0.10580146, 0.54291755, -0.06625827, -0.3755...",1
74665,1,nvidia did delay by weeks,"[nvidia, did, delay, by, weeks]","[nvidia, delay, weeks]","[-0.22863172, 0.35537758, -0.100245036, -0.062...",3
74666,1,nvidia really delayed the several weeks,"[nvidia, really, delayed, the, several, weeks]","[nvidia, really, delayed, several, weeks]","[-0.17035049, 0.45445055, -0.05678586, -0.3756...",1
74667,1,nvidia really only delayed the flight weeks,"[nvidia, really, only, delayed, the, flight, w...","[nvidia, really, delayed, flight, weeks]","[-0.08468987, 0.43954343, -0.059893716, -0.321...",1
74668,1,nvidia really delayed the next weeks,"[nvidia, really, delayed, the, next, weeks]","[nvidia, really, delayed, next, weeks]","[0.060755454, 0.5819391, -0.08134587, -0.24678...",1
74669,3,let no elim go unnoticed nvidia highlights aut...,"[let, no, elim, go, unnoticed, nvidia, highlig...","[let, elim, go, unnoticed, nvidia, highlights,...","[0.05612147, 0.28515637, -0.17512518, -0.14771...",3
74670,3,t let elim go unnoticed nvidia highlights auto...,"[t, let, elim, go, unnoticed, nvidia, highligh...","[let, elim, go, unnoticed, nvidia, highlights,...","[0.08029704, 0.19982113, -0.14802913, -0.05101...",3


In [25]:
# Streamlit is already installed in this environment.
# If you encounter a ModuleNotFoundError, please restart the Colab runtime (Runtime -> Restart runtime).
# You do not need to run !pip install streamlit again after restarting.

In [ ]:
import gradio as gr
import pandas as pd
import re
import numpy as np
import emoji
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# NLTK data downloads (ensure these are run once in your notebook environment if not already)
# nltk.download("punkt")
# nltk.download("stopwords")

# --- 1. Load Data and Re-initialize Preprocessing Components (Run once) ---
# Re-load the data and fit the LabelEncoder to ensure consistent mapping
# In a real deployed app, you would save and load the LabelEncoder and models.
# For this Colab environment, we'll re-create the necessary components.

try:
    df = pd.read_csv("/content/twitter_training.csv") # Changed to twitter_training.csv
    # Drop columns as done in the main notebook (cell JTqyTGOtDLvt)
    df.drop(["Borderlands","2401"], inplace=True,axis=1)
    # Rename columns as done in the main notebook (cell FyTifsabDbmt)
    df.rename(columns={
        "Positive": "sentiment",
        "im getting on borderlands and i will murder you all ,": "tweet"
    }, inplace=True)
except FileNotFoundError:
    print("CSV file not found. Please ensure 'twitter_training.csv' is in '/content/'.")
    # Create an empty DataFrame to prevent further errors if file is missing
    df = pd.DataFrame(columns=['sentiment', 'tweet'])

# Clean text function (from your notebook)
def clean_text(text):
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+',' ',text).strip()
    return text

# Only proceed with preprocessing if DataFrame is not empty
if not df.empty:
    df['tweet'] = df['tweet'].apply(clean_text)
    df["tokenized_tweet"] = df["tweet"].apply(word_tokenize)

    stop_words = set(stopwords.words("english"))
    df["fillterd_tweets"] = df["tokenized_tweet"].apply(
        lambda words : [word for word in words if word not in stop_words]
    )

    # Re-train Word2Vec model
    corpus = df['fillterd_tweets'].tolist()
    w2v_model = Word2Vec(
        sentences=corpus,
        vector_size=200,
        window=5,
        min_count=2,
        workers=4
    )

    # Re-create sentence vector function
    def get_sentence_vector(word_list, model):
        vectors = [
            model.wv[word] for word in word_list if word in model.wv
        ]
        if vectors:
            return np.mean(vectors, axis=0)
        else:
            return np.zeros(model.vector_size)

    df['sentence_vector'] = df['fillterd_tweets'].apply(lambda x: get_sentence_vector(x, w2v_model))

    # Re-fit LabelEncoder
    la = LabelEncoder()
    df["sentiment"] = la.fit_transform(df["sentiment"])

    # Prepare data for model training (as done in notebook)
    x = df["sentence_vector"]
    y = df["sentiment"]

    # Ensure there's enough data for splitting and training
    if len(x) > 1 and len(y) > 1 and len(x) == len(y):
        xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.15, random_state=42)
        X_train_full = np.stack(xtrain.values)
        X_test_final = np.stack(xtest.values)
        X_train_final, X_val, y_train_final, y_val = train_test_split(
            X_train_full, ytrain, test_size=0.15, random_state=42, stratify=ytrain
        )

        # Re-initialize and train the KNN model
        knn_model = KNeighborsClassifier()
        knn_model.fit(X_train_final, y_train_final)
    else:
        knn_model = None
        print("Warning: Insufficient data for model training. Model will not be available.")
        # Initialize necessary objects to avoid NameError if df was empty
        w2v_model = None
        la = None
        stop_words = set()
else:
    knn_model = None
    w2v_model = None
    la = None
    stop_words = set()
    print("Warning: DataFrame is empty, model training and preprocessing skipped.")


# --- Gradio Prediction Function ---
def predict_sentiment_gradio(user_input_text):
    if not user_input_text or user_input_text.strip() == "":
        return "Please enter some text for sentiment analysis."

    if knn_model is None or w2v_model is None or la is None:
        return "Application not initialized: Model or preprocessing components are missing due to earlier errors."

    # Preprocess User Input
    cleaned_input = clean_text(user_input_text)
    tokenized_input = word_tokenize(cleaned_input)
    filtered_input = [word for word in tokenized_input if word not in stop_words]

    # Get Sentence Vector
    # Ensure get_sentence_vector is defined and callable
    # (moved definition outside this function for scope, if it wasn't already)
    vectors = [
        w2v_model.wv[word] for word in filtered_input if word in w2v_model.wv
    ]
    if vectors:
        input_vector = np.mean(vectors, axis=0)
    else:
        input_vector = np.zeros(w2v_model.vector_size) # Use model's vector_size

    # Reshape for model prediction (model expects 2D array)
    input_vector = input_vector.reshape(1, -1)

    # Make Prediction
    prediction_encoded = knn_model.predict(input_vector)

    # Decode Prediction
    prediction_label = la.inverse_transform(prediction_encoded)

    return f"Predicted Sentiment: {prediction_label[0]}"


# --- Gradio App Layout ---
# Only launch Gradio if the model and components were successfully trained/initialized
if knn_model is not None and w2v_model is not None and la is not None:
    interface = gr.Interface(
        fn=predict_sentiment_gradio,
        inputs=gr.Textbox(lines=2, placeholder="Type your sentence here...", label="Enter your text here:"),
        outputs=gr.Textbox(label="Predicted Sentiment"),
        title="Sentiment Analysis Web App",
        description="Enter text below to get its predicted sentiment using a pre-trained K-Nearest Neighbors model."
    )
    interface.launch(debug=True) # debug=True can be useful for debugging
else:
    print("Gradio application not launched due to previous errors in data loading or model training.")

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://83e3cf941fd6826086.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
